# Zim-my — ADTC 2026 Fine-tuning

In [ ]:
# Set environment variables BEFORE any torch imports
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import shutil

# Delete unsloth compiled cache to avoid fused loss errors
cache_dir = "unsloth_compiled_cache"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"✓ Deleted {cache_dir}")

from datasets import load_from_disk
from transformers import AutoTokenizer, Trainer
from unsloth import FastLanguageModel
from transformers import TrainingArguments, DataCollatorForLanguageModeling

# Fix paths — notebook is in notebooks/, data is in ../data/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_PROCESSED = os.path.join(PROJECT_ROOT, 'data', 'processed')

print(f"Project root: {PROJECT_ROOT}")
print(f"Data processed dir: {DATA_PROCESSED}")

# Check GPU
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU available! Please run this on a GPU instance.")

## 1. Load Base Model with 4-bit Quantization

In [ ]:
# Model configuration
model_name = "Qwen/Qwen2.5-3B-Instruct"
max_seq_length = 512  # Reduced aggressively to fit T4 14GB
dtype = None  # Auto-detect (Float16 for V100, BFloat16 for Ampere+)
load_in_4bit = True  # QLoRA - unsloth will quantize on the fly

# Clear GPU cache before loading
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
import gc
gc.collect()

print(f"Loading model: {model_name}")
print(f"Max sequence length: {max_seq_length}")
print(f"4-bit quantization: {load_in_4bit}")

# Model already downloaded via ModelScope to this path
local_model_path = "/mnt/workspace/models/Qwen/Qwen2.5-3B-Instruct"

if not os.path.exists(local_model_path):
    print(f"Model not found at {local_model_path}")
    print("Download it first with:")
    print("python -c \"from modelscope import snapshot_download; snapshot_download('Qwen/Qwen2.5-3B-Instruct', cache_dir='/mnt/workspace/models')\"")
    raise FileNotFoundError(f"Model not found at {local_model_path}")
else:
    print(f"✓ Model found at {local_model_path}")

# Load model and tokenizer from local path
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=local_model_path,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    device_map="cuda:0",
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"\n✓ Model loaded successfully")

## 2. Add LoRA Adapters

In [ ]:
# LoRA configuration
lora_rank = 16  # Reduced aggressively to fit T4 14GB
lora_alpha = 32  # Keep alpha = 2 * rank
lora_dropout = 0.05
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

print(f"LoRA Configuration:")
print(f"  Rank: {lora_rank}")
print(f"  Alpha: {lora_alpha}")
print(f"  Dropout: {lora_dropout}")
print(f"  Target modules: {target_modules}")

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=target_modules,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Saves 50% memory
    random_state=42,
)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✓ LoRA adapters added")
print(f"Trainable parameters: {trainable_params / 1e6:.2f}M ({trainable_params/total_params*100:.2f}%)")

## 3. Load and Prepare Training Data

In [ ]:
# Load processed datasets
train_dataset = load_from_disk(os.path.join(DATA_PROCESSED, 'train'))
val_dataset = load_from_disk(os.path.join(DATA_PROCESSED, 'val'))

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")
print(f"\nSample training example:")
print(train_dataset[0])

In [ ]:
# Format prompt template for Qwen2.5-Instruct
alpaca_prompt = """<|im_start|>system
You are Zim-my, an AI assistant developed by Michael Mlungisi Nkomo, an artificial intelligence engineer from Zimbabwe. You specialize in Zimbabwean agriculture and can communicate in Shona and English.
<|im_end|>
<|im_start|>user
{instruction}
{input}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Handle empty input
        if input_text and input_text.strip():
            input_section = f"\n{input_text}"
        else:
            input_section = ""
        
        text = alpaca_prompt.format(
            instruction=instruction,
            input=input_section,
            output=output
        ) + EOS_TOKEN
        texts.append(text)
    
    return {"text": texts}

# Apply formatting
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print(f"\n✓ Formatted datasets")
print(f"\nSample formatted text:")
print(train_dataset[0]["text"][:500])

# Tokenize the text data for standard Trainer
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",
    )

train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

print(f"\n✓ Tokenized datasets")

## 4. Configure Training

In [ ]:
# Training configuration
output_dir = "models/lora/qwen2.5-3b-zim-my"
num_train_epochs = 3
per_device_train_batch_size = 1  # Minimum to fit T4 14GB
gradient_accumulation_steps = 32  # Effective batch size = 32
learning_rate = 2e-4
weight_decay = 0.01
warmup_ratio = 0.03
lr_scheduler_type = "cosine"
fp16 = not torch.cuda.is_bf16_supported()
bf16 = torch.cuda.is_bf16_supported()
logging_steps = 10
save_steps = 500
eval_steps = 500
save_total_limit = 1

print("Training Configuration:")
print(f"  Output directory: {output_dir}")
print(f"  Epochs: {num_train_epochs}")
print(f"  Batch size: {per_device_train_batch_size}")
print(f"  Gradient accumulation: {gradient_accumulation_steps}")
print(f"  Effective batch size: {per_device_train_batch_size * gradient_accumulation_steps}")
print(f"  Learning rate: {learning_rate}")
print(f"  Warmup ratio: {warmup_ratio}")
print(f"  FP16: {fp16}, BF16: {bf16}")

# Create output directory
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Initialize standard Trainer (not SFTTrainer) to avoid fused loss issues
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal language modeling
)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        lr_scheduler_type=lr_scheduler_type,
        fp16=fp16,
        bf16=bf16,
        logging_steps=logging_steps,
        save_steps=save_steps,
        eval_steps=eval_steps,
        save_total_limit=save_total_limit,
        optim="adamw_8bit",
        seed=42,
        report_to="none",
        eval_strategy="no",
        load_best_model_at_end=False,
        remove_unused_columns=False,
    ),
    train_dataset=train_dataset,
    eval_dataset=None,
    data_collator=data_collator,
)

print("\n✓ Trainer initialized")

## 5. Start Training

In [ ]:
# Memory stats before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_gpu_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU: {gpu_stats.name}")
print(f"GPU Memory: {max_gpu_memory:.1f} GB")
print(f"Reserved Memory: {start_gpu_memory:.1f} GB")
print(f"\nStarting training...")

In [ ]:
# Clear cache before training
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Run training
trainer_stats = trainer.train()

print(f"\n✓ Fine-tuning complete!")
print(f"LoRA adapters saved to: {output_dir}/final")

In [ ]:
# Memory stats after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_gpu_memory * 100, 1)
lora_percentage = round(used_memory_for_lora / max_gpu_memory * 100, 1)

print(f"Memory Usage:")
print(f"  Peak reserved: {used_memory:.1f} GB ({used_percentage}%)")
print(f"  LoRA training: {used_memory_for_lora:.1f} GB ({lora_percentage}%)")

## 6. Save LoRA Adapters

In [ ]:
# Save final LoRA adapters
final_lora_path = f"{output_dir}/final"
model.save_pretrained(final_lora_path)
tokenizer.save_pretrained(final_lora_path)

print(f"✓ Saved LoRA adapters to: {final_lora_path}")
print(f"\nNext: Run notebooks/03_quantize.ipynb to merge and quantize")

## 7. Test the Fine-tuned Model

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

# Test prompts
test_prompts = [
    ("What are the main crops grown in Zimbabwe?", ""),
    ("Pindura mubvunzo uyu: Chii chinonzi photosynthesis?", ""),
    ("Explain crop rotation and its benefits", ""),
]

print("Testing fine-tuned model:\n")
for instruction, input_text in test_prompts:
    if input_text:
        input_section = f"\n{input_text}"
    else:
        input_section = ""
    
    prompt = alpaca_prompt.format(
        instruction=instruction,
        input=input_section,
        output=""
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract assistant response
    if "<|im_start|>assistant" in response:
        response = response.split("<|im_start|>assistant")[-1].strip()
    
    print(f"Q: {instruction}")
    print(f"A: {response[:200]}...")
    print("-" * 80)

## Summary

Fine-tuning complete! The LoRA adapters have been saved to `models/lora/qwen2.5-3b-laptoplelm/final`.

### Next Steps:
1. Open `notebooks/03_quantize.ipynb`
2. Merge LoRA adapters with base model
3. Export to GGUF format (Q4_K_M quantization)
4. Test inference speed on CPU

### Training Results:
- Final training loss: Check the logs above
- Validation loss: Check the logs above
- Memory usage: Check the stats above
- Training time: Check the stats above